# PaceAlgo ML — Notebook 11: Phase 1 Feature Ablation + Stability

**Strenge Methodik:**
1. **5 Ablation-Experimente** auf identischem Walk-Forward-Split:
   - `00_baseline` = 15 SHAP-Top-Features aus NB 08
   - `10_baseline+SMC` = +18 Market-Structure-Features
   - `20_baseline+Session` = +12 Session/Vol-Features
   - `30_baseline+HTF` = +12 HTF×LTF-Interaction-Features
   - `40_all_combined` = alle ~55 Features
2. **4 Metriken pro Experiment × Tier**: PF, WR, **Expectancy** (R/Trade), **OOS-Stability** (PF-Konsistenz pro Jahr)
3. **Asset-Split mit Gewinner-Config**: Combined vs FX-only vs Gold-only
4. **SHAP-Filter**: Tote Features konsequent rauswerfen
5. **Final**: kleinstes robustes Feature-Set

Grundregel: kein Feature bleibt ohne SHAP-Relevanz **UND** PF-Lift in Ablation.

## 1. Setup

In [ ]:
import sys, os
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    DATA_RAW = Path(PROJECT_ROOT) / 'data_cache' / 'raw'
    DATA_PROCESSED = Path('/content/processed')
    DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    ARTIFACTS = Path(PROJECT_ROOT) / 'artifacts'
    REPORTS_DIR = ARTIFACTS / 'reports'
    os.chdir(PROJECT_ROOT)
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED.glob('*.parquet'))) < (len(list(DRIVE_BACKUP.glob('*.parquet'))) if DRIVE_BACKUP.exists() else 0):
        !rsync -ah {DRIVE_BACKUP}/ {DATA_PROCESSED}/
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    from core.config import DATA_RAW, DATA_PROCESSED, ARTIFACTS_REPORTS
    REPORTS_DIR = ARTIFACTS_REPORTS

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]

In [ ]:
!pip install -q lightgbm shap 2>&1 | tail -1

import json
import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from core.config import (
    FX_SYMBOLS, METAL_SYMBOLS, PRIMARY_TIMEFRAMES, DEV_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END, HTF_CONTEXT_TIMEFRAMES,
)
from core.features import (
    compute_features, attach_macro, attach_htf_context,
    compute_smc_features, compute_session_features, compute_htf_interactions,
)
from core.features.engineer import atr as atr_fn
from core.train import walk_forward_split, binary_label_for_long

R_VALUE = 1.5
TRAINING_SYMBOLS = FX_SYMBOLS + METAL_SYMBOLS

## 2. Build Extended Feature Set

In [ ]:
EXTENDED_DIR = Path('/content/processed_v2') if IS_COLAB else (DATA_PROCESSED.parent / 'processed_v2')
EXTENDED_DIR.mkdir(parents=True, exist_ok=True)

def load_ohlcv(symbol, tf):
    p = DATA_RAW / f'{symbol}_{tf}.parquet'
    return pd.read_parquet(p) if p.exists() else None

macro_path = DATA_RAW / 'macro_daily.parquet'
macro = pd.read_parquet(macro_path) if macro_path.exists() else pd.DataFrame()

for symbol in tqdm(TRAINING_SYMBOLS, desc='Extended features'):
    htf_feats = {}
    for htf in HTF_CONTEXT_TIMEFRAMES:
        d = load_ohlcv(symbol, htf)
        if d is None or d.empty:
            htf_feats[htf] = pd.DataFrame()
        else:
            htf_feats[htf] = compute_features(d)

    for tf in PRIMARY_TIMEFRAMES:
        out_path = EXTENDED_DIR / f'{symbol}_{tf}_extended.parquet'
        if out_path.exists():
            continue
        ohlcv = load_ohlcv(symbol, tf)
        if ohlcv is None or ohlcv.empty:
            continue
        base = compute_features(ohlcv)
        base = attach_htf_context(base, htf_feats.get('1h', pd.DataFrame()), htf_feats.get('4h', pd.DataFrame()))
        base = attach_macro(base, macro)
        atr14 = atr_fn(ohlcv['high'], ohlcv['low'], ohlcv['close'], 14).values
        ema_align = base['ema_alignment'].fillna(0).values
        smc = compute_smc_features(ohlcv, atr14, ema_align)
        sess = compute_session_features(ohlcv, atr14)
        inter = compute_htf_interactions(base)
        ext = pd.concat([base, smc, sess, inter], axis=1)
        label_path = DATA_PROCESSED / f'labels_{symbol}_{tf}_R{R_VALUE}.parquet'
        if label_path.exists():
            labels = pd.read_parquet(label_path)
            ext = ext.join(labels[['label']], how='inner')
        ext['symbol'] = symbol
        ext['timeframe'] = tf
        ext.to_parquet(out_path, compression='zstd')
    htf_feats.clear()

print('Extended feature files ready.')
for f in sorted(EXTENDED_DIR.glob('*.parquet')):
    print(f'  {f.name}: {f.stat().st_size / 1024**2:.1f} MB')

## 3. Helper Functions — train + eval with PF/WR/Expectancy/Stability

In [ ]:
PARAMS_PINE = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 7, 'max_depth': 3, 'min_data_in_leaf': 200,
    'learning_rate': 0.05, 'num_iterations': 30, 'lambda_l2': 0.5,
    'feature_fraction': 0.85, 'bagging_fraction': 0.85, 'bagging_freq': 5,
    'is_unbalance': True, 'verbose': -1, 'n_jobs': -1,
}

WIN_R = R_VALUE
LOSS_R = 1.0

def tier_metrics_from_labels(labels: pd.Series):
    wins = int((labels == 1).sum())
    losses = int((labels == -1).sum())
    neutrals = int((labels == 0).sum())
    total = wins + losses + neutrals
    pf = (wins * WIN_R) / (losses * LOSS_R) if losses > 0 else (float('inf') if wins > 0 else 0.0)
    wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
    expectancy_R = (wins * WIN_R - losses * LOSS_R) / total if total > 0 else 0.0
    return {'n_trades': total, 'wins': wins, 'losses': losses, 'neutrals': neutrals,
             'win_rate': wr, 'profit_factor': pf, 'expectancy_R': expectancy_R}


def load_extended(symbol, tf):
    p = EXTENDED_DIR / f'{symbol}_{tf}_extended.parquet'
    return pd.read_parquet(p) if p.exists() else None


def stack_extended(symbols, tfs, drop_holdout):
    drop = set(drop_holdout or [])
    frames = []
    for s in symbols:
        if s in drop:
            continue
        for tf in tfs:
            d = load_extended(s, tf)
            if d is None or d.empty:
                continue
            frames.append(d)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, axis=0).sort_index()


def train_and_evaluate(df_full, feature_cols, exp_label='?'):
    """Train Pine-budget model, compute per-tier metrics + per-year stability + per-asset breakdown."""
    feat_present = [f for f in feature_cols if f in df_full.columns]
    df_c = df_full.dropna(subset=feat_present + ['label'])
    if len(df_c) < 5000:
        return None
    train_df, val_df, test_df = walk_forward_split(df_c, TRAIN_END, VAL_END)
    if len(train_df) < 1000 or len(val_df) < 100 or len(test_df) < 100:
        return None

    X_tr = train_df[feat_present]; y_tr = binary_label_for_long(train_df['label'])
    X_vl = val_df[feat_present]; y_vl = binary_label_for_long(val_df['label'])
    X_ts = test_df[feat_present]

    td = lgb.Dataset(X_tr, label=y_tr)
    vd = lgb.Dataset(X_vl, label=y_vl, reference=td)
    model = lgb.train(PARAMS_PINE, td, num_boost_round=30,
                       valid_sets=[vd], valid_names=['val'],
                       callbacks=[lgb.log_evaluation(period=0)])

    val_proba = model.predict(X_vl)
    test_proba = model.predict(X_ts)

    # Tier cutoffs from VAL only (honest)
    vs = np.sort(val_proba)[::-1]
    cutoff_std = float(vs[max(1, int(len(vs) * 0.10) - 1)])
    cutoff_high = float(vs[max(1, int(len(vs) * 0.03) - 1)])
    cutoff_prem = float(vs[max(1, int(len(vs) * 0.01) - 1)])

    # Apply to TEST set per tier
    tiers = {}
    for name, cutoff in [('Standard', cutoff_std), ('High', cutoff_high), ('Premium', cutoff_prem)]:
        mask = test_proba >= cutoff
        sub_labels = test_df['label'].iloc[mask.nonzero()[0]]
        m = tier_metrics_from_labels(sub_labels)
        m['cutoff'] = cutoff
        tiers[name] = m

    # Per-year stability for Premium tier
    test_df_copy = test_df.copy()
    test_df_copy['proba'] = test_proba
    test_df_copy['year'] = test_df_copy.index.year
    yearly_pf = {}
    for year, sub in test_df_copy.groupby('year'):
        mask = sub['proba'] >= cutoff_prem
        sub_labels = sub.loc[mask, 'label']
        if len(sub_labels) < 30:
            continue
        m = tier_metrics_from_labels(sub_labels)
        yearly_pf[int(year)] = m['profit_factor']

    pf_values = [v for v in yearly_pf.values() if not np.isinf(v) and v > 0]
    if len(pf_values) >= 2:
        stability_cv = float(np.std(pf_values) / np.mean(pf_values))
        n_years_positive = sum(1 for v in pf_values if v > 1.0)
    else:
        stability_cv = float('nan')
        n_years_positive = len(pf_values)

    # Per-asset breakdown for Premium tier
    per_asset = {}
    for sym in test_df_copy['symbol'].unique():
        mask = (test_df_copy['symbol'] == sym) & (test_df_copy['proba'] >= cutoff_prem)
        sub_labels = test_df_copy.loc[mask, 'label']
        if len(sub_labels) < 30:
            continue
        per_asset[sym] = tier_metrics_from_labels(sub_labels)

    return {
        'label': exp_label,
        'feature_cols': feat_present,
        'n_features': len(feat_present),
        'n_train': len(train_df),
        'n_test': len(test_df),
        'tiers': tiers,
        'yearly_pf_premium': yearly_pf,
        'stability_cv': stability_cv,
        'n_years_positive': n_years_positive,
        'per_asset_premium': per_asset,
        'model': model,
        'val_proba': val_proba,
        'test_proba': test_proba,
        'test_df': test_df,
    }


print('Helper functions ready.')

## 4. Feature Ablation — 5 Experiments on Combined FX+Gold

In [ ]:
# Feature group definitions
BASELINE = [
    'hour_sin', 'dist_to_swing_low_atr', 'htf_1h_rsi_14', 'realized_vol_20',
    'atr_percentile_100', 'atr_pct', 'dist_to_swing_high_atr', 'volume_z_score',
    'ema_20_slope_atr', 'hour_cos', 'momentum_composite', 'rvol_20',
    'adx_14', 'ema_20_dist_atr', 'htf_1h_atr_percentile_100',
]

SMC_FEATS = [
    'sweep_up_recent', 'sweep_down_recent', 'bars_since_sweep_up', 'bars_since_sweep_down',
    'eqhigh_present', 'eqlow_present', 'dist_to_sh_atr_conf', 'dist_to_sl_atr_conf',
    'bos_up_strict', 'bos_down_strict', 'choch_to_down', 'choch_to_up',
    'fvg_bull_active', 'fvg_bear_active', 'dist_to_bull_fvg_atr', 'dist_to_bear_fvg_atr',
    'fvg_bull_size_atr', 'fvg_bear_size_atr',
]

SESSION_FEATS = [
    'in_asia', 'in_london', 'in_ny', 'in_london_ny_killzone',
    'in_asia_london_overlap', 'in_us_open_killzone', 'in_london_open_killzone',
    'is_fx_market_open', 'vol_expansion_ratio', 'vol_expanding', 'vol_contracting',
    'bars_since_vol_spike',
]

HTF_INTER_FEATS = [
    'htf_ltf_agree_bull', 'htf_ltf_agree_bear', 'htf_ltf_counter_trend',
    'htf_ltf_alignment_score', 'ltf_rsi_minus_htf_rsi',
    'both_rsi_oversold', 'both_rsi_overbought', 'vol_pct_diff_htf',
    'both_high_vol', 'both_low_vol', 'pullback_in_bull', 'pullback_in_bear',
]

ABLATION_CONFIGS = {
    '00_baseline':         BASELINE,
    '10_base+SMC':         BASELINE + SMC_FEATS,
    '20_base+Session':     BASELINE + SESSION_FEATS,
    '30_base+HTF_inter':   BASELINE + HTF_INTER_FEATS,
    '40_all_combined':     BASELINE + SMC_FEATS + SESSION_FEATS + HTF_INTER_FEATS,
}

df_combined = stack_extended(TRAINING_SYMBOLS, PRIMARY_TIMEFRAMES, DEV_HOLDOUT_SYMBOLS)
print(f'Combined FX+Gold dataset: {df_combined.shape}')

ablation_results = {}
for name, feats in ABLATION_CONFIGS.items():
    print(f'\nRunning {name} ({len(feats)} features)...')
    res = train_and_evaluate(df_combined, feats, exp_label=name)
    if res is None:
        print(f'  -> SKIPPED (insufficient data)')
        continue
    ablation_results[name] = res
    prem = res['tiers']['Premium']
    high = res['tiers']['High']
    std_ = res['tiers']['Standard']
    print(f'  Premium: PF {prem["profit_factor"]:.3f}, WR {prem["win_rate"]*100:.1f}%, ExpR {prem["expectancy_R"]:+.3f}, n {prem["n_trades"]}')
    print(f'  High:    PF {high["profit_factor"]:.3f}, WR {high["win_rate"]*100:.1f}%, ExpR {high["expectancy_R"]:+.3f}, n {high["n_trades"]}')
    print(f'  Stability CV: {res["stability_cv"]:.3f}, years positive: {res["n_years_positive"]}')

## 5. Ablation Summary Table

In [ ]:
rows = []
for name, res in ablation_results.items():
    for tier in ['Standard', 'High', 'Premium']:
        t = res['tiers'][tier]
        rows.append({
            'experiment': name,
            'tier': tier,
            'n_features': res['n_features'],
            'n_trades': t['n_trades'],
            'WR': t['win_rate'],
            'PF': t['profit_factor'],
            'ExpR': t['expectancy_R'],
            'stability_CV_premium': res['stability_cv'],
            'years_positive_premium': res['n_years_positive'],
        })
ablation_df = pd.DataFrame(rows)

# Pivot for easy reading
print('=== Profit Factor by experiment x tier ===')
display(ablation_df.pivot_table(index='experiment', columns='tier', values='PF').round(4)[['Standard', 'High', 'Premium']])

print('\n=== Win Rate ===')
display(ablation_df.pivot_table(index='experiment', columns='tier', values='WR').round(4)[['Standard', 'High', 'Premium']])

print('\n=== Expectancy per Trade (R-multiples) ===')
display(ablation_df.pivot_table(index='experiment', columns='tier', values='ExpR').round(4)[['Standard', 'High', 'Premium']])

print('\n=== Premium Stability ===')
stab = ablation_df[ablation_df['tier'] == 'Premium'][['experiment', 'stability_CV_premium', 'years_positive_premium', 'n_trades']]
display(stab.round(4))

## 6. Lift Analysis — Does each Feature Group Actually Help?

In [ ]:
if '00_baseline' in ablation_results:
    base_pf = ablation_results['00_baseline']['tiers']['Premium']['profit_factor']
    base_expR = ablation_results['00_baseline']['tiers']['Premium']['expectancy_R']
    base_wr = ablation_results['00_baseline']['tiers']['Premium']['win_rate']

    print(f'Baseline Premium: PF {base_pf:.3f}, WR {base_wr*100:.1f}%, ExpR {base_expR:+.4f}')
    print()
    print(f'{"Group":<25s} {"PF lift":<10s} {"WR lift":<10s} {"ExpR lift":<12s} {"Verdict":<20s}')
    print('-' * 80)
    for name, res in ablation_results.items():
        if name == '00_baseline':
            continue
        prem = res['tiers']['Premium']
        pf_lift = prem['profit_factor'] - base_pf
        wr_lift = prem['win_rate'] - base_wr
        expR_lift = prem['expectancy_R'] - base_expR
        # Verdict
        if pf_lift > 0.05 and expR_lift > 0.01:
            verdict = 'POSITIVE (keep)'
        elif pf_lift > 0.0 and expR_lift > 0.0:
            verdict = 'marginal'
        elif pf_lift < -0.05:
            verdict = 'NEGATIVE (drop)'
        else:
            verdict = 'neutral'
        print(f'{name:<25s} {pf_lift:+.3f}     {wr_lift*100:+.1f}%      {expR_lift:+.4f}      {verdict}')

## 7. SHAP Analysis on Combined Model (Identify Truly Useful Features)

In [ ]:
best_ablation_name = '40_all_combined' if '40_all_combined' in ablation_results else list(ablation_results.keys())[-1]
best_res = ablation_results[best_ablation_name]
model = best_res['model']
test_df_full = best_res['test_df']
feat_cols = best_res['feature_cols']

rng = np.random.default_rng(42)
sample_idx = rng.choice(len(test_df_full), size=min(10_000, len(test_df_full)), replace=False)
X_sample = test_df_full.iloc[sample_idx][feat_cols]

explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(X_sample)
mean_abs = np.abs(sv).mean(axis=0)
shap_table = pd.DataFrame({
    'feature': feat_cols, 'mean_abs_shap': mean_abs,
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

def classify(f):
    if f in BASELINE: return 'Baseline'
    if f in SMC_FEATS: return 'SMC'
    if f in SESSION_FEATS: return 'Session/Vol'
    if f in HTF_INTER_FEATS: return 'HTF-Interaction'
    return 'Other'
shap_table['group'] = shap_table['feature'].apply(classify)

print('Top 30 features by SHAP:')
display(shap_table.head(30))

group_summary = shap_table.groupby('group')['mean_abs_shap'].agg(['sum', 'mean', 'count']).sort_values('sum', ascending=False)
print('\nSHAP contribution per group:')
display(group_summary)

dead = shap_table[shap_table['mean_abs_shap'] < 1e-5]
print(f'\nDead features (zero SHAP): {len(dead)} of {len(shap_table)}')
if len(dead) > 0 and len(dead) < 40:
    print('  ' + ', '.join(dead['feature'].tolist()))

## 8. Find Smallest Robust Feature Set (SHAP-Filter)

In [ ]:
filter_results = {}
for top_n in [10, 12, 15, 18, 20, 25]:
    selected = list(shap_table.head(top_n)['feature'])
    res = train_and_evaluate(df_combined, selected, exp_label=f'SHAP-Top-{top_n}')
    if res is None:
        continue
    filter_results[f'SHAP_top_{top_n}'] = res
    prem = res['tiers']['Premium']
    print(f'SHAP-Top-{top_n}: Premium PF {prem["profit_factor"]:.3f}, ExpR {prem["expectancy_R"]:+.4f}, n {prem["n_trades"]}, stability CV {res["stability_cv"]:.3f}')

# Combined table
rows = []
for name, res in {**ablation_results, **filter_results}.items():
    for tier in ['Standard', 'High', 'Premium']:
        t = res['tiers'][tier]
        rows.append({'experiment': name, 'tier': tier, 'n_features': res['n_features'],
                      'PF': t['profit_factor'], 'WR': t['win_rate'], 'ExpR': t['expectancy_R'],
                      'stability_CV': res['stability_cv']})
all_summary = pd.DataFrame(rows)
premium_view = all_summary[all_summary['tier'] == 'Premium'].sort_values('PF', ascending=False)
print('\nAll experiments ranked by Premium PF:')
display(premium_view[['experiment', 'n_features', 'PF', 'WR', 'ExpR', 'stability_CV']].round(4))

## 9. Asset Split with Winning Feature Set

In [ ]:
# Choose winning config: best Premium PF AND at least 1000 Premium trades AND stability CV < 0.5
candidates = premium_view[(premium_view['stability_CV'].fillna(99) < 0.5)].copy()
candidates = candidates.sort_values('PF', ascending=False)
if not candidates.empty:
    winning_name = candidates.iloc[0]['experiment']
else:
    winning_name = premium_view.iloc[0]['experiment']
print(f'Winning feature configuration: {winning_name}')
winning_res = ablation_results.get(winning_name) or filter_results.get(winning_name)
winning_features = winning_res['feature_cols']
print(f'  Features ({len(winning_features)}): {winning_features}')

# Now apply to asset splits
asset_splits = {
    'combined_FX+Gold': TRAINING_SYMBOLS,
    'FX_only':           FX_SYMBOLS,
    'Gold_only':         METAL_SYMBOLS,
}
asset_results = {}
for asym_label, symbols in asset_splits.items():
    df_split = stack_extended(symbols, PRIMARY_TIMEFRAMES, DEV_HOLDOUT_SYMBOLS)
    if df_split.empty:
        continue
    res = train_and_evaluate(df_split, winning_features, exp_label=asym_label)
    if res is None:
        continue
    asset_results[asym_label] = res
    prem = res['tiers']['Premium']
    print(f'\n{asym_label}: Premium PF {prem["profit_factor"]:.3f}, WR {prem["win_rate"]*100:.1f}%, n {prem["n_trades"]:,}, stability CV {res["stability_cv"]:.3f}, years+ {res["n_years_positive"]}')
    print('  Per-asset Premium breakdown:')
    for sym, m in res['per_asset_premium'].items():
        print(f'    {sym}: PF {m["profit_factor"]:.3f}, WR {m["win_rate"]*100:.1f}%, n {m["n_trades"]:,}')

## 10. Save Best Config for NB 12 (Model Battery)

In [ ]:
# Pick BEST setup: across feature ablations AND asset splits, find max Premium PF
all_results = {**ablation_results, **filter_results, **{f'asset_{k}': v for k, v in asset_results.items()}}
scoreboard = []
for name, res in all_results.items():
    prem = res['tiers']['Premium']
    if prem['n_trades'] < 200:  # discard low-volume
        continue
    scoreboard.append({
        'name': name,
        'n_features': res['n_features'],
        'premium_pf': prem['profit_factor'],
        'premium_wr': prem['win_rate'],
        'premium_expR': prem['expectancy_R'],
        'premium_n': prem['n_trades'],
        'stability_cv': res['stability_cv'],
        'years_positive': res['n_years_positive'],
    })
sb_df = pd.DataFrame(scoreboard).sort_values('premium_pf', ascending=False)
print('Final scoreboard (Premium tier):')
display(sb_df.round(4))

# Save best config
best_row = sb_df.iloc[0]
best_name = best_row['name']
best_result = all_results[best_name]
best_config = {
    'experiment_name': best_name,
    'feature_cols': best_result['feature_cols'],
    'n_features': best_result['n_features'],
    'premium_pf_oos': float(best_row['premium_pf']),
    'premium_wr_oos': float(best_row['premium_wr']),
    'premium_expR_oos': float(best_row['premium_expR']),
    'stability_cv': float(best_result['stability_cv']) if not np.isnan(best_result['stability_cv']) else None,
    'is_asset_split': best_name.startswith('asset_'),
    'asset_scope': best_name.replace('asset_', '') if best_name.startswith('asset_') else 'combined_FX+Gold',
}
out_path = REPORTS_DIR / 'phase1_best_config.json'
with open(out_path, 'w') as f:
    json.dump(best_config, f, indent=2)
print(f'\nBest config saved: {out_path}')
print(json.dumps(best_config, indent=2))

## 11. Final Verdict

In [ ]:
print('=' * 70)
print('PHASE 1 EVALUATION — FINAL VERDICT')
print('=' * 70)
print(f'\nReference (NB 08 baseline): Premium PF ~1.79')
print(f'Best Phase-1 config: {best_name}')
print(f'  Features used: {best_config["n_features"]}')
print(f'  Premium PF: {best_config["premium_pf_oos"]:.4f}')
print(f'  Premium WR: {best_config["premium_wr_oos"]*100:.1f}%')
print(f'  Premium ExpR/trade: {best_config["premium_expR_oos"]:+.4f}')
print(f'  Stability CV: {best_config["stability_cv"]:.3f}' if best_config["stability_cv"] else '  Stability: insufficient data')

phase1_lift = best_config['premium_pf_oos'] - 1.79
print(f'\nLift vs NB 08 baseline: {phase1_lift:+.4f}')

if best_config['premium_pf_oos'] >= 1.9:
    print('\n  -> PHASE 1 TARGET MET. Proceed to NB 12 (Model Battery)')
elif phase1_lift > 0.05:
    print('\n  -> Solid Phase 1 lift. Proceed to NB 12 to see if better model arch pushes us over 1.9')
elif phase1_lift > 0.0:
    print('\n  -> Marginal Phase 1 lift. Phase 2 (Cross-Asset, Meta-Labeling) needed')
else:
    print('\n  -> No lift. New features did NOT help. Investigate SHAP group breakdown.')

print('\nNext: open NB 12 to test XGBoost / CatBoost / Voting on this config.')
print('=' * 70)